


# Aprendizaje No Supervisado DiploDatos 2025
## Georgina Flesia y Laura Alonso Alemany

# Análisis FIFA 2018 - Clustering Reduced Dataset

##  <span style="font-family: Blippo, fantasy; font-size: 23px; font-weight: bold; letter-spacing: 3px; color: #b30024">Inicialización-del-entorno</span>
Empezamos cargando algunas herramientas para cargar los datos y manipularlos.

In [ ]:
import numpy as np
import pandas as pd
import os
import sys
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import re#, wget, os
np.set_printoptions(legacy='1.25')

Este es un grupo de datos reducido y escogido, no tiene los nombre de los jugadores. Arma los clusters ovalados, ideal para mezcla de gaussianas?
Tiene las siguientes etiquetas de posiciones

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/DiploDatos/AprendizajeNOSupervisado/master/2024/2025/male_players.csv")
# df = pd.read_csv("./sample_data/male_players.csv")
df.columns

In [ ]:
# Eliminamos las columnas innecesarias
del df['Unnamed: 0']
del df['Unnamed: 0.1']
df.columns


# About this Dataset

The dataset provides comprehensive information on FC 25 players, focusing on their in-game ratings, attributes, and additional statistics.
It is derived from EA SPORTS Website using web scraping as shown in the notebook here
Here’s a detailed breakdown of the columns in the dataset along with their meanings:

    Rank: Player’s ranking based on overall rating (OVR) within the FC 25 group.
    Name: The full name of the player.
    Height: The player’s height (*)
    Weight: The player’s weight (*)
    Position: The primary position the player plays on the field
    Alternative positions: Other positions the player is capable of playing effectively.
    Age: The player’s age.
    Nation: The country the player represents in international competitions.
    League: The football league in which the player currently plays.
    Team: The club team the player is part of.
    Play style: Specific gameplay traits or tendencies that define the player’s behavior and skillset on the field (e.g., "Quick Step", "Finesse Shot").
    URL: A link to the player's detailed profile.

Player Attributes

    Acceleration: The player’s ability to reach maximum speed quickly. (*)
    Sprint Speed: The top speed the player can achieve when sprinting.(*)
    Positioning: The player's awareness and positioning in attack.(*)
    Finishing: The player’s ability to convert scoring chances into goals.(*)
    Shot Power: The strength of the player’s shots on goal. (*)
    Long Shots: The accuracy and power of shots taken from outside the penalty area. (*)
    Volleys: The player’s ability to strike the ball cleanly from mid-air.(**)
    Penalties: The player's skill at taking penalty kicks.(**)

Passing and Vision

    Vision: The player's ability to make accurate passes and see plays develop. (*)
    Crossing: The ability to deliver accurate crosses from wide areas. (*)
    Free Kick Accuracy: The player’s precision when taking free kicks.
    Short Passing: The accuracy and skill in making short-distance passes.(*)
    Long Passing: The ability to deliver accurate long-range passes.(*)
    Curve: The player’s ability to bend the ball during passes or shots.

Dribbling and Agility

    Dribbling: The player’s ball control and ability to maneuver in tight spaces.(*)
    Agility: How quickly and smoothly the player can change direction.(*)
    Balance: The player’s stability and ability to stay on their feet under pressure.(*)
    Reactions: The player’s responsiveness to unpredictable events during the game.(*)
    Ball Control: How well the player controls the ball while moving.(*)

Mentality and Defense

    Composure: The player’s calmness under pressure.(*)
    Interceptions: The player’s ability to read and intercept passes.(*)
    Heading Accuracy: The player's precision when attempting to head the ball.(*)
    Defensive Awareness (Def Awareness): The player’s positioning and ability to anticipate defensive situations.(*)
    Standing Tackle: The player’s ability to win the ball with a standing tackle.(*)
    Sliding Tackle: The skill and accuracy of the player’s sliding tackles.(*)

Physical Attributes

    Jumping: The player’s ability to jump high during headers or challenges. (*)
    Stamina: The player’s endurance and ability to perform at a high level throughout the match.(*)
    Strength: The player’s physical power and ability to win physical challenges.(*)
    Aggression: The player’s determination and intensity in winning challenges and duels.(*)

Technical Skills

    Weak foot: The player’s proficiency with their non-dominant foot (rated from 1 to 5 stars).(*)
    Skill moves: The player’s ability to perform advanced dribbling moves (rated from 1 to 5 stars).(*)
    Preferred foot: Indicates whether the player prefers using their left or right foot.

Goalkeeping Attributes (if applicable)

    GK Diving: The goalkeeper’s ability to dive and make saves.(*)
    GK Handling: The goalkeeper’s skill in catching or holding onto the ball.(*)
    GK Kicking: The accuracy and power of the goalkeeper’s kicks when distributing the ball.(*)
    GK Positioning: The goalkeeper’s ability to position themselves effectively during defensive situations.(*)
    GK Reflexes: The goalkeeper’s quickness in reacting to shots.(*)


In [ ]:
df.columns

OVR → Valoración General (Overall Rating): la nota global del jugador, entre 0 y 99.

PAC → Ritmo (Pace): mide la velocidad y aceleración del jugador.

SHO → Tiro (Shooting): refleja la capacidad de disparo y finalización.

PAS → Pase (Passing): indica la precisión y calidad en los pases.

DRI → Regate (Dribbling): mide la habilidad de conducción y control del balón.

DEF → Defensa (Defending): representa la capacidad defensiva (marcaje, entradas, intercepciones).

PHY → Físico (Physical): refleja la fuerza, resistencia y agresividad del jugador.

OVR (Valoración General / Overall Rating)

Escala: 0–99

Cómo se mide: es un cálculo interno del juego que combina varios atributos individuales, ponderados según la posición del jugador.

Ejemplo: un delantero tiene el OVR más influenciado por PAC, SHO y DRI, mientras que un defensa depende más de DEF y PHY.

🔹 PAC (Ritmo / Pace)

Escala: 0–99

Cómo se mide: promedio ponderado de:

Sprint Speed (Velocidad máxima)

Acceleration (Aceleración)

🔹 SHO (Tiro / Shooting)

Escala: 0–99

Cómo se mide: combinación de subatributos:

Finishing (Definición)

Shot Power (Potencia de tiro)

Long Shots (Tiros lejanos)

Volleys (Voleas)

Penalties (Penales)

🔹 PAS (Pase / Passing)

Escala: 0–99

Cómo se mide: mezcla de precisión en:

Short Passing (Pase corto)

Long Passing (Pase largo)

Crossing (Centros)

Vision (Visión de juego)

Free Kick Accuracy (Precisión en tiros libres)

Curve (Efecto en el balón)

🔹 DRI (Regate / Dribbling)

Escala: 0–99

Cómo se mide: combinación de:

Ball Control (Control del balón)

Agility (Agilidad)

Balance (Equilibrio)

Reactions (Reacciones)

Dribbling (Habilidad de regate)

Composure (Compostura)

🔹 DEF (Defensa / Defending)

Escala: 0–99

Cómo se mide: pondera atributos como:

Interceptions (Intercepciones)

Heading Accuracy (Precisión de cabeza)

Marking (Marcaje)

Standing Tackle (Entrada normal)

Sliding Tackle (Entrada al suelo)

🔹 PHY (Físico / Physical)

Escala: 0–99

Cómo se mide: promedio de:

Strength (Fuerza)

Stamina (Resistencia)

Jumping (Salto)

Aggression (Agresividad)

👉 En resumen: todas estas métricas se expresan en una escala de 0 a 99, y se calculan a partir de subatributos más detallados que también aparecen en el dataset del FIFA.

In [ ]:
df.Position.unique()

Las posiciones de fútbol indicadas son:
- ST: delantero.

- CDM: medio centro defensivo.

- CAM: medio centro ofensivo.

- LW: extremo izquierdo.

- CM: medio centro.

- GK: portero.

- CB: defensa central.

- RW: extremo derecho.

- LB: lateral izquierdo.

- LM: medio izquierdo.

- RB: lateral derecho.

- RM: medio derecho.


# Realizamos un agrupamiento mas general para poder realizar el analisis.


In [ ]:
# ST = Delantero
# CM = Medio Campo
# DB = Defensor
# GK = Arquero

group_position = {
    'ST': 'ST',
    'CDM': 'CM',
    'CAM': 'CM',
    'LW': 'ST',
    'CM': 'CM',
    'GK': 'GK',
    'CB': 'CB',
    'RW': 'ST',
    'LB': 'CB',
    'LM': 'CM',
    'RB': 'CB',
    'RM': 'CM'
}
df['General_Position'] = df['Position'].map(group_position)
df.General_Position.unique()

In [ ]:
df.Position.value_counts()

In [ ]:
df.General_Position.value_counts()

In [ ]:
df.Nation.value_counts()

In [ ]:
df.Team.value_counts()

# Curacion de datos

In [ ]:
sns.heatmap(df==None, cbar=False, cmap='viridis')

In [ ]:
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')

In [ ]:
sns.heatmap(df==0, cbar=False, cmap='viridis')

# Matriz de Correlacion entre las variables

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(df.select_dtypes(include='number').corr(), cmap="vlag")
plt.show()

Vamos a rellenar con 0 los valores NULL

In [ ]:
df.fillna(0, inplace=True)

In [ ]:
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')

- Pair plot for average attributes to see general attributions beetween diferent positions.

In [ ]:
sns.pairplot(df[['OVR','PAC','SHO','PAS','DRI','DEF','PHY','Position']], hue='Position',palette='tab10')
plt.show()

- Pair plot for player attributes

In [ ]:
data = df[['Acceleration','Sprint Speed','Positioning','Finishing','Shot Power','Long Shots','Volleys','Position']].dropna()

# Crear el gráfico de densidad
plt.figure(figsize=(10, 6))
sns.kdeplot(data, shade=True, color="skyblue", linewidth=2)

# Personalizar el gráfico
plt.title("Distribución Gaussiana", fontsize=14)
plt.xlabel("Columnas")
plt.ylabel("Densidad")
plt.grid(True)
plt.show()


# Vamos a elegir las variables que pensamos que mas destacan a los jugadores usando la clasificacion de GK, DB, CM y ST, y de esas caracteristicas vamos a quedarnos solamente con las 4 primeras de cada grupo para realizar un analisis exploratorio. Los atributos escogidos son marcados con (*).

|    GK    |     DB    |     CM    |     ST    |
| -------- | --------  | --------  | --------  |
| Vision(*) |  Dribbling | Strength | Short Passing |
| Crossing(*) | Interceptions(*) | Stamina(*) | Dribbling(*) |
| Short Passing | Heading Accuracy  | Aggression | Agility(*) |
| Long Passing | Def Awareness(*) | Interceptions | Balance |
| Agility | Standing Tackle(*) | Standing Tackle | Heading Accuracy(*) |
| Reactions(*) | Sliding Tackle(*) | Balance(*) | Jumping |
| Ball Control(*) | Aggression | Reactions | Aggression |
| GK Diving | Jumping | Short Passing(*) | Skill moves |
| GK Handling |  | Long Passing(*) | Acceleration(*) |
| GK Kicking |  | Free Kick Accuracy | Free Kick Accuracy |
| GK Positioning |  |  | Finishing |
| GK Reflexes |  |  |  |


In [ ]:
df_principal_properties = df[[
    'Vision','Crossing','Reactions','Ball Control',
    'Interceptions','Def Awareness','Standing Tackle','Sliding Tackle',
    'Stamina','Balance','Short Passing','Long Passing',
    'Dribbling','Agility','Heading Accuracy','Acceleration',
    'Position','General_Position']]

In [ ]:
sns.pairplot(df_principal_properties, hue='Position',palette='tab10')
plt.show()

In [ ]:
sns.pairplot(df_principal_properties, hue='General_Position',palette='tab10')
plt.show()

In [ ]:
df_principal_properties.columns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player Attributes by Position')

sns.scatterplot(data=df_principal_properties, x='Dribbling', y='Heading Accuracy', hue='Position',palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('Dribbling vs Heading Accuracy')

sns.scatterplot(data=df_principal_properties, x='Dribbling', y='Acceleration', hue='Position',palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('Dribbling vs Acceleration')

sns.scatterplot(data=df_principal_properties, x='Dribbling', y='Interceptions', hue='Position',palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Dribbling vs Interceptions')

sns.scatterplot(data=df_principal_properties, x='Interceptions', y='Crossing', hue='Position',palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Interceptions vs Crossing')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
df_principal_properties.columns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player Attributes by Position')

sns.scatterplot(data=df_principal_properties, x='Heading Accuracy', y='Def Awareness', hue='Position',palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('Heading Accuracy vs. Def Awareness')

sns.scatterplot(data=df_principal_properties, x='Heading Accuracy', y='Sliding Tackle', hue='Position',palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('Heading Accuracy vs. Sliding Tackle')

sns.scatterplot(data=df_principal_properties, x='Heading Accuracy', y='Interceptions', hue='Position',palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Heading Accuracy vs. Interceptions')

sns.scatterplot(data=df_principal_properties, x='Heading Accuracy', y='Standing Tackle', hue='Position',palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Heading Accuracy vs. Standing Tackle')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player Attributes by General_Position')

sns.scatterplot(data=df_principal_properties, x='Dribbling', y='Heading Accuracy', hue='General_Position',palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('Dribbling vs Heading Accuracy')

sns.scatterplot(data=df_principal_properties, x='Dribbling', y='Acceleration', hue='General_Position',palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('Dribbling vs Acceleration')

sns.scatterplot(data=df_principal_properties, x='Dribbling', y='Interceptions', hue='General_Position',palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Dribbling vs Interceptions')

sns.scatterplot(data=df_principal_properties, x='Interceptions', y='Crossing', hue='General_Position',palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Interceptions vs Crossing')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player Attributes by General_Position')

sns.scatterplot(data=df_principal_properties, x='Heading Accuracy', y='Def Awareness', hue='General_Position',palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('Def Awareness vs. Heading Accuracy')

sns.scatterplot(data=df_principal_properties, x='Heading Accuracy', y='Sliding Tackle', hue='General_Position',palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('Heading Accuracy vs. Sliding Tackle')

sns.scatterplot(data=df_principal_properties, x='Heading Accuracy', y='Interceptions', hue='General_Position',palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Heading Accuracy vs. Interceptions')

sns.scatterplot(data=df_principal_properties, x='Heading Accuracy', y='Standing Tackle', hue='General_Position',palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Heading Accuracy vs. Standing Tackle')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player (CB, CM) Attributes by General Position')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','CM'])], x='Acceleration', y='Dribbling', hue='General_Position', ax=axes[0, 0])
axes[0, 0].set_title('Accelaration vs Dribbling')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','CM'])], x='Heading Accuracy', y='Dribbling', hue='General_Position', ax=axes[0, 1])
axes[0, 1].set_title('Heading Accuracy vs Dribbling')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','CM'])],  x='Interceptions', y='Crossing', hue='General_Position', ax=axes[1, 0])
axes[1, 0].set_title('Interceptions vs Crossing')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','CM'])], x='Dribbling', y='Interceptions', hue='General_Position', ax=axes[1, 1])
axes[1, 1].set_title('Dribbling vs Interceptions')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player Attributes (CB, ST) by General Position')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','ST'])], x='Acceleration', y='Dribbling', hue='General_Position', ax=axes[0, 0])
axes[0, 0].set_title('Accelaration vs Dribbling')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','ST'])], x='Heading Accuracy', y='Dribbling', hue='General_Position', ax=axes[0, 1])
axes[0, 1].set_title('Heading Accuracy vs Dribbling')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','ST'])],  x='Interceptions', y='Crossing', hue='General_Position', ax=axes[1, 0])
axes[1, 0].set_title('Interceptions vs Crossing')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','ST'])], x='Dribbling', y='Interceptions', hue='General_Position', ax=axes[1, 1])
axes[1, 1].set_title('Dribbling vs Interceptions')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player Attributes (CM, ST) by General Position')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CM','ST'])], x='Acceleration', y='Dribbling', hue='General_Position', ax=axes[0, 0])
axes[0, 0].set_title('Accelaration vs Dribbling')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CM','ST'])], x='Heading Accuracy', y='Dribbling', hue='General_Position', ax=axes[0, 1])
axes[0, 1].set_title('Heading Accuracy vs Dribbling')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CM','ST'])],  x='Interceptions', y='Crossing', hue='General_Position', ax=axes[1, 0])
axes[1, 0].set_title('Interceptions vs Crossing')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CM','ST'])], x='Dribbling', y='Interceptions', hue='General_Position', ax=axes[1, 1])
axes[1, 1].set_title('Dribbling vs Interceptions')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player (CB, CM) Attributes by General Position')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','CM'])], x='Heading Accuracy', y='Def Awareness', hue='General_Position', ax=axes[0, 0])
axes[0, 0].set_title('Heading Accuracy vs. Def Awareness')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','CM'])], x='Heading Accuracy', y='Sliding Tackle', hue='General_Position', ax=axes[0, 1])
axes[0, 1].set_title('Heading Accuracy vs Sliding Tackle')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','CM'])], x='Heading Accuracy' ,y='Interceptions', hue='General_Position', ax=axes[1, 0])
axes[1, 0].set_title('Heading Accuracy vs. Interceptions')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','CM'])], x='Heading Accuracy', y='Standing Tackle', hue='General_Position', ax=axes[1, 1])
axes[1, 1].set_title('Heading Accuracy vs Standing Tackle')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player (CB, ST) Attributes by General Position')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','ST'])], x='Heading Accuracy', y='Def Awareness', hue='General_Position', ax=axes[0, 0])
axes[0, 0].set_title('Heading Accuracy vs. Def Awareness')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','ST'])], x='Heading Accuracy', y='Sliding Tackle', hue='General_Position', ax=axes[0, 1])
axes[0, 1].set_title('Heading Accuracy vs Sliding Tackle')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','ST'])], x='Heading Accuracy' ,y='Interceptions', hue='General_Position', ax=axes[1, 0])
axes[1, 0].set_title('Heading Accuracy vs. Interceptions')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CB','ST'])], x='Heading Accuracy', y='Standing Tackle', hue='General_Position', ax=axes[1, 1])
axes[1, 1].set_title('Heading Accuracy vs Standing Tackle')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player (CM, ST) Attributes by General Position')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CM','ST'])], x='Heading Accuracy', y='Def Awareness', hue='General_Position', ax=axes[0, 0])
axes[0, 0].set_title('Heading Accuracy vs. Def Awareness')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CM','ST'])], x='Heading Accuracy', y='Sliding Tackle', hue='General_Position', ax=axes[0, 1])
axes[0, 1].set_title('Heading Accuracy vs Sliding Tackle')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CM','ST'])], x='Heading Accuracy' ,y='Interceptions', hue='General_Position', ax=axes[1, 0])
axes[1, 0].set_title('Heading Accuracy vs. Interceptions')

sns.scatterplot(data=df_principal_properties[df_principal_properties.General_Position.isin(['CM','ST'])], x='Heading Accuracy', y='Standing Tackle', hue='General_Position', ax=axes[1, 1])
axes[1, 1].set_title('Heading Accuracy vs Standing Tackle')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

# Conclusion parcial sobre el analisis de datos

Observamos que existen variables que mejor caracterizan a los grupos de jugadores escogidos y permiten una clusterizacion sin la necesidad de aplicar algoritmos.

# Embbeding y escaleo de las variables escogidas


In [ ]:
prefered_attributes = [
    'Acceleration','Sprint Speed','Positioning','Finishing','Shot Power',
    'Long Shots','Volleys','Vision','Crossing','Short Passing','Long Passing','Dribbling',
    'Agility','Balance','Reactions','Ball Control','Composure','Interceptions','Heading Accuracy',
    'Def Awareness','Standing Tackle','Sliding Tackle','Jumping','Stamina',
    'Strength','Aggression','Skill moves','GK Diving','GK Handling','GK Kicking',
    'GK Positioning','GK Reflexes']

df_cluster = df[prefered_attributes].reset_index(drop=True)
df_cluster.head()


In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import MinMaxScaler, StandardScaler

feature_dict_general_position = list(df_cluster.T.to_dict().values())

# DictVectorizer
vec = DictVectorizer()
feature_matrix_general_position = vec.fit_transform(feature_dict_general_position)

# MinMaxScaler
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler_feature_matrix_general_position = scaler.fit_transform(feature_matrix_general_position.toarray())

In [ ]:
feature_dict_general_position[1].keys()

# **KMeans applications**


## Numero de cluster escogidos 4

In [ ]:
from sklearn.cluster import KMeans,MeanShift

# Número de clusters buscado
n_clust = 4

km_4 = KMeans(n_clusters=n_clust,random_state=5)
km_4.fit(scaler_feature_matrix_general_position) #utiliza todas las habilidades: 33 dimensiones

# Etiquetas asignadas por el algoritmo
clusters = km_4.labels_
print('Suma de los cuadrados de las distancias al centro de cada cluster=Inertia= ', km_4.inertia_)
df['kmeans_4'] = km_4.labels_ #clusters
print('Kmeans encontró: ', max(km_4.labels_)+1, 'clusters, nosotros forzamos la cantidad')

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer

# Instantiate the clustering model and visualizer
visualizer = SilhouetteVisualizer(km_4)

visualizer.fit(scaler_feature_matrix_general_position)
visualizer.show()        # Finalize and render the figure

## Numero de cluster escogidos 5

In [ ]:
# Número de clusters buscado
n_clust = 5

km_5 = KMeans(n_clusters=n_clust,random_state=6)
km_5.fit(scaler_feature_matrix_general_position) #utiliza todas las habilidades: 33 dimensiones

# Etiquetas asignadas por el algoritmo
clusters = km_5.labels_
print('Suma de los cuadrados de las distancias al centro de cada cluster=Inertia= ', km_5.inertia_)
df['kmeans_5'] = km_5.labels_ #clusters
print('Kmeans encontró: ', max(km_5.labels_)+1, 'clusters, nosotros forzamos la cantidad')

In [ ]:
# Instantiate the clustering model and visualizer
visualizer = SilhouetteVisualizer(km_5)

visualizer.fit(scaler_feature_matrix_general_position)
visualizer.show()        # Finalize and render the figure

## Numero de cluster escogidos 6

In [ ]:
# Número de clusters buscado
n_clust = 6

km_6 = KMeans(n_clusters=n_clust,random_state=7)
km_6.fit(scaler_feature_matrix_general_position) #utiliza todas las habilidades: 33 dimensiones

# Etiquetas asignadas por el algoritmo
clusters = km_6.labels_
print('Suma de los cuadrados de las distancias al centro de cada cluster=Inertia= ', km_6.inertia_)
df['kmeans_6'] = km_6.labels_ #clusters
print('Kmeans encontró: ', max(km_6.labels_)+1, 'clusters, nosotros forzamos la cantidad')

In [ ]:
# Instantiate the clustering model and visualizer
visualizer = SilhouetteVisualizer(km_6)

visualizer.fit(scaler_feature_matrix_general_position)
visualizer.show()        # Finalize and render the figure

## Numero de cluster escogidos 7

In [ ]:
# Número de clusters buscado
n_clust = 7

km_7 = KMeans(n_clusters=n_clust,random_state=8)
km_7.fit(scaler_feature_matrix_general_position) #utiliza todas las habilidades: 33 dimensiones

# Etiquetas asignadas por el algoritmo
clusters = km_7.labels_
print('Suma de los cuadrados de las distancias al centro de cada cluster=Inertia= ', km_7.inertia_)
df['kmeans_7'] = km_7.labels_ #clusters
print('Kmeans encontró: ', max(km_7.labels_)+1, 'clusters, nosotros forzamos la cantidad')

In [ ]:
# Instantiate the clustering model and visualizer
visualizer = SilhouetteVisualizer(km_7)

visualizer.fit(scaler_feature_matrix_general_position)
visualizer.show()        # Finalize and render the figure

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player Attributes by kmeans clusters')

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='kmeans_4',palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('Heading Accuracy vs. Def Awareness')

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='kmeans_5',palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('Heading Accuracy vs. Def Awareness')

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='kmeans_6',palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Heading Accuracy vs. Def Awareness')

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='kmeans_7',palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Heading Accuracy vs. Def Awareness')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player Attributes by kmeans clusters')

sns.scatterplot(data=df, x='Dribbling', y='Interceptions', hue='kmeans_4',palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('Dribbling vs Interceptions')

sns.scatterplot(data=df, x='Dribbling', y='Interceptions', hue='kmeans_5',palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('Dribbling vs Interceptions')

sns.scatterplot(data=df, x='Dribbling', y='Interceptions', hue='kmeans_6',palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Dribbling vs Interceptions')

sns.scatterplot(data=df, x='Dribbling', y='Interceptions', hue='kmeans_7',palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Dribbling vs Interceptions')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 15))
fig.suptitle('Attributes by kmeans clusters')

sns.countplot(df,x="kmeans_4",hue="Position",palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('Positions attribution for n_cluster=4')

sns.countplot(df,x="kmeans_5",hue="Position",palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('Positions attribution for n_cluster=5')

sns.countplot(df,x="kmeans_6",hue="Position",palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Positions attribution for n_cluster=6')

sns.countplot(df,x="kmeans_7",hue="Position",palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Positions attribution for n_cluster=7')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Attributes by kmeans clusters')

sns.countplot(df,x="kmeans_4",hue="General_Position",palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('General_Position attribution for n_cluster=4')

sns.countplot(df,x="kmeans_5",hue="General_Position",palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('General_Position attribution for n_cluster=5')

sns.countplot(df,x="kmeans_6",hue="General_Position",palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('General_Position attribution for n_cluster=6')

sns.countplot(df,x="kmeans_7",hue="General_Position",palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('General_Position attribution for n_cluster=7')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

# **DBSCAN**

## eps = 0.7

In [ ]:
from sklearn.cluster import DBSCAN

dbScan = DBSCAN(eps=0.7, min_samples=10)
dbScan.fit(scaler_feature_matrix_general_position)

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(dbScan.labels_)) - (1 if -1 in dbScan.labels_ else 0)
n_noise_ = list(dbScan.labels_).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)

df['DBSCAN_07'] = dbScan.labels_

In [ ]:
fig, axes = plt.subplots(3, figsize=(18, 15))

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='DBSCAN_07',palette='tab10',ax=axes[0])
axes[0].set_title('Heading Accuracy vs. Def Awareness')


sns.countplot(df,x="DBSCAN_07",hue="Position",palette='tab10', ax=axes[1])
axes[1].set_title('Positions attribution for ep=0.7')

sns.countplot(df,x="DBSCAN_07",hue="General_Position",palette='tab10', ax=axes[2])
axes[2].set_title('General_Positions attribution for ep=0.7')

plt.show()

## eps=0.8

In [ ]:
dbScan = DBSCAN(eps=0.8, min_samples=6)
dbScan.fit(scaler_feature_matrix_general_position)
# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(dbScan.labels_)) - (1 if -1 in dbScan.labels_ else 0)
n_noise_ = list(dbScan.labels_).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)
df['DBSCAN_08'] = dbScan.labels_

In [ ]:
fig, axes = plt.subplots(3, figsize=(18, 15))

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='DBSCAN_08',palette='tab10',ax=axes[0])
axes[0].set_title('Heading Accuracy vs. Def Awareness')


sns.countplot(df,x="DBSCAN_08",hue="Position",palette='tab10', ax=axes[1])
axes[1].set_title('Positions attribution for ep=0.8')

sns.countplot(df,x="DBSCAN_08",hue="General_Position",palette='tab10', ax=axes[2])
axes[2].set_title('General_Positions attribution for ep=0.8')

plt.show()

## eps = 0.9

In [ ]:
dbScan = DBSCAN(eps=0.9, min_samples=5)
dbScan.fit(scaler_feature_matrix_general_position)
# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(dbScan.labels_)) - (1 if -1 in dbScan.labels_ else 0)
n_noise_ = list(dbScan.labels_).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)
df['DBSCAN_09'] = dbScan.labels_

In [ ]:
fig, axes = plt.subplots(3, figsize=(18, 15))

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='DBSCAN_09',palette='tab10',ax=axes[0])
axes[0].set_title('Heading Accuracy vs. Def Awareness')


sns.countplot(df,x="DBSCAN_09",hue="Position",palette='tab10', ax=axes[1])
axes[1].set_title('Positions attribution for ep=0.9')

sns.countplot(df,x="DBSCAN_09",hue="General_Position",palette='tab10', ax=axes[2])
axes[2].set_title('General_Positions attribution for ep=0.9')

plt.show()

CONCLUSION: hicimos pruebas con DBSCAN y el resultado fue que no se separa correctamente la cantidad de cluster en comparacion con K-MEANS.

In [ ]:
import itertools

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy import linalg

from sklearn import mixture

color_iter = itertools.cycle(["navy", "c", "cornflowerblue", "gold", "darkorange"])


def plot_results(X, Y_, means, covariances, index, title):
    splot = plt.subplot(2, 1, 1 + index)
    for i, (mean, covar, color) in enumerate(zip(means, covariances, color_iter)):
        v, w = linalg.eigh(covar)
        v = 2.0 * np.sqrt(2.0) * np.sqrt(v)
        u = w[0] / linalg.norm(w[0])
        # as the DP will not use every component it has access to
        # unless it needs it, we shouldn't plot the redundant
        # components.
        if not np.any(Y_ == i):
            continue
        plt.scatter(X[Y_ == i, 0], X[Y_ == i, 1], 0.8, color=color)

        # Plot an ellipse to show the Gaussian component
        angle = np.arctan(u[1] / u[0])
        angle = 180.0 * angle / np.pi  # convert to degrees
        ell = mpl.patches.Ellipse(mean, v[0], v[1], angle=180.0 + angle, color=color)
        ell.set_clip_box(splot.bbox)
        ell.set_alpha(0.5)
        splot.add_artist(ell)

    plt.xlim(-9.0, 5.0)
    plt.ylim(-3.0, 6.0)
    plt.xticks(())
    plt.yticks(())
    plt.title(title)


# Reduccion de dimensionalidad (PCA)

Primero vamos a implementar la transformacion *DictVectorizer* para poder trabajar tanto con datos numericos como categoricos.


In [ ]:
# PCA
from sklearn.decomposition import PCA
pca = PCA(n_components=5)
pca_position = pca.fit_transform(scaler_feature_matrix_general_position)
print(f"shape: {pca_position.shape}")
print(f"num_components 5:{pca.explained_variance_ratio_}")
print('Escogiendo las 5 componentes tenemos determinada el aprox. el 85% de los datos.')

#pca = PCA(n_components=10)
#proyected_feature_matrix_position = pca.fit_transform(scaler_feature_matrix_general_position)
#proyected_feature_matrix_position.shape
#print(f"num_components 10:{pca.explained_variance_ratio_}")

#pca = PCA(n_components=15)
#proyected_feature_matrix_position = pca.fit_transform(scaler_feature_matrix_general_position)
#proyected_feature_matrix_position.shape
#print(f"num_components 15:{pca.explained_variance_ratio_}")

#pca = PCA(n_components=20)
#proyected_feature_matrix_position = pca.fit_transform(scaler_feature_matrix_general_position)
#proyected_feature_matrix_position.shape
#print(f"num_components 20:{pca.explained_variance_ratio_}")

In [ ]:
df['x1_pca']= pca_position[:,0]
df['x2_pca']= pca_position[:,1]
df['x3_pca']= pca_position[:,2]
df['x4_pca']= pca_position[:,3]
df['x5_pca']= pca_position[:,4]

# Kmeans con PCA + Scaler

In [ ]:
from yellowbrick.cluster import KElbowVisualizer
model = KMeans()
fig = KElbowVisualizer(model,k=(1,10))
fig.fit(pca_position)
fig.show()

## n_cluster = 3 para PCA + Scaler

In [ ]:
from sklearn.cluster import KMeans,MeanShift

# Número de clusters buscado
n_clust = 3

km_3_pca = KMeans(n_clusters=n_clust,random_state=25)
km_3_pca.fit(pca_position) #utiliza todas las habilidades: 33 dimensiones

# Etiquetas asignadas por el algoritmo
clusters = km_3_pca.labels_
print('Suma de los cuadrados de las distancias al centro de cada cluster=Inertia= ', km_3_pca.inertia_)
df['kmeans_3_pca'] = km_3_pca.labels_ #clusters
print('Kmeans encontró: ', max(km_3_pca.labels_)+1, 'clusters, nosotros forzamos la cantidad')

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer

# Instantiate the clustering model and visualizer
visualizer = SilhouetteVisualizer(km_3_pca)

visualizer.fit(pca_position)
visualizer.show()        # Finalize and render the figure

In [ ]:
with plt.style.context('dark_background'):
    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="General_Position",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)

    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="kmeans_3_pca",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)
plt.show()

## n_cluster = 4 para PCA + Scaler

In [ ]:
from sklearn.cluster import KMeans,MeanShift

# Número de clusters buscado
n_clust = 4

km_4_pca = KMeans(n_clusters=n_clust,random_state=425)
km_4_pca.fit(pca_position) #utiliza todas las habilidades: 33 dimensiones

# Etiquetas asignadas por el algoritmo
clusters = km_4_pca.labels_
print('Suma de los cuadrados de las distancias al centro de cada cluster=Inertia= ', km_4_pca.inertia_)
df['kmeans_4_pca'] = km_4_pca.labels_ #clusters
print('Kmeans encontró: ', max(km_4_pca.labels_)+1, 'clusters, nosotros forzamos la cantidad')

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer

# Instantiate the clustering model and visualizer
visualizer = SilhouetteVisualizer(km_4_pca)

visualizer.fit(pca_position)
visualizer.show()        # Finalize and render the figure

In [ ]:
with plt.style.context('dark_background'):
    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="General_Position",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)

    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="kmeans_4_pca",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)
plt.show()

## n_cluster = 5 para Scaler + PCA

In [ ]:
from sklearn.cluster import KMeans,MeanShift

# Número de clusters buscado
n_clust = 5

km_5_pca = KMeans(n_clusters=n_clust,random_state=525)
km_5_pca.fit(pca_position) #utiliza todas las habilidades: 33 dimensiones

# Etiquetas asignadas por el algoritmo
clusters = km_5_pca.labels_
print('Suma de los cuadrados de las distancias al centro de cada cluster=Inertia= ', km_5_pca.inertia_)
df['kmeans_5_pca'] = km_5_pca.labels_ #clusters
print('Kmeans encontró: ', max(km_5_pca.labels_)+1, 'clusters, nosotros forzamos la cantidad')

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer

# Instantiate the clustering model and visualizer
visualizer = SilhouetteVisualizer(km_5_pca)

visualizer.fit(pca_position)
visualizer.show()        # Finalize and render the figure

In [ ]:
with plt.style.context('dark_background'):
    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="General_Position",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)

    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="kmeans_5_pca",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)
plt.show()

## n_cluster = 6 para Scaler + PCA



In [ ]:
from sklearn.cluster import KMeans,MeanShift

# Número de clusters buscado
n_clust = 6

km_6_pca = KMeans(n_clusters=n_clust,random_state=625)
km_6_pca.fit(pca_position) #utiliza todas las habilidades: 33 dimensiones

# Etiquetas asignadas por el algoritmo
clusters = km_6_pca.labels_
print('Suma de los cuadrados de las distancias al centro de cada cluster=Inertia= ', km_6_pca.inertia_)
df['kmeans_6_pca'] = km_6_pca.labels_ #clusters
print('Kmeans encontró: ', max(km_6_pca.labels_)+1, 'clusters, nosotros forzamos la cantidad')

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer

# Instantiate the clustering model and visualizer
visualizer = SilhouetteVisualizer(km_6_pca)

visualizer.fit(pca_position)
visualizer.show()        # Finalize and render the figure

In [ ]:
with plt.style.context('dark_background'):
    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="General_Position",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)

    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="kmeans_6_pca",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)
plt.show()

## n_cluster = 7, para Scaler + PCA


In [ ]:
from sklearn.cluster import KMeans,MeanShift

# Número de clusters buscado
n_clust = 7

km_7_pca = KMeans(n_clusters=n_clust,random_state=725)
km_7_pca.fit(pca_position) #utiliza todas las habilidades: 33 dimensiones

# Etiquetas asignadas por el algoritmo
clusters = km_7_pca.labels_
print('Suma de los cuadrados de las distancias al centro de cada cluster=Inertia= ', km_7_pca.inertia_)
df['kmeans_7_pca'] = km_7_pca.labels_ #clusters
print('Kmeans encontró: ', max(km_7_pca.labels_)+1, 'clusters, nosotros forzamos la cantidad')

In [ ]:
from yellowbrick.cluster import SilhouetteVisualizer

# Instantiate the clustering model and visualizer
visualizer = SilhouetteVisualizer(km_7_pca)

visualizer.fit(pca_position)
visualizer.show()        # Finalize and render the figure

In [ ]:
with plt.style.context('dark_background'):
    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="General_Position",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)

    g = sns.relplot(x='x1_pca',
                    y='x2_pca',
                    hue="kmeans_7_pca",
                    kind="scatter",
                    #style = 'PrefPos',
                    data=df,
                    alpha = 0.7)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Scatter Plots of Player Attributes by kmeans clusters with Scaler + PCA attributes')

sns.scatterplot(data=df, x='Dribbling', y='Interceptions', hue='kmeans_4_pca',palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('Dribbling vs Interceptions')

sns.scatterplot(data=df, x='Dribbling', y='Interceptions', hue='kmeans_5_pca',palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('Dribbling vs Interceptions')

sns.scatterplot(data=df, x='Dribbling', y='Interceptions', hue='kmeans_6_pca',palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('Dribbling vs Interceptions')

sns.scatterplot(data=df, x='Dribbling', y='Interceptions', hue='kmeans_7_pca',palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('Dribbling vs Interceptions')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Attributes by kmeans clusters for PCA')

sns.countplot(df,x="kmeans_4_pca",hue="General_Position",palette='tab10', ax=axes[0, 0])
axes[0, 0].set_title('General_Position attribution for n_cluster=4')

sns.countplot(df,x="kmeans_5_pca",hue="General_Position",palette='tab10', ax=axes[0, 1])
axes[0, 1].set_title('General_Position attribution for n_cluster=5')

sns.countplot(df,x="kmeans_6_pca",hue="General_Position",palette='tab10', ax=axes[1, 0])
axes[1, 0].set_title('General_Position attribution for n_cluster=6')

sns.countplot(df,x="kmeans_7_pca",hue="General_Position",palette='tab10', ax=axes[1, 1])
axes[1, 1].set_title('General_Position attribution for n_cluster=7')

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

# **DBSCAN con PCA**

## eps=0.7

In [ ]:
from sklearn.cluster import DBSCAN

dbScan = DBSCAN(eps=0.7, min_samples=10)
dbScan.fit(pca_position)

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(dbScan.labels_)) - (1 if -1 in dbScan.labels_ else 0)
n_noise_ = list(dbScan.labels_).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)

df['DBSCAN_07_pca'] = dbScan.labels_

In [ ]:
fig, axes = plt.subplots(3, figsize=(18, 15))

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='DBSCAN_07_pca',palette='tab10',ax=axes[0])
axes[0].set_title('Heading Accuracy vs. Def Awareness')


sns.countplot(df,x="DBSCAN_07_pca",hue="Position",palette='tab10', ax=axes[1])
axes[1].set_title('Positions attribution for ep=0.7')

sns.countplot(df,x="DBSCAN_07_pca",hue="General_Position",palette='tab10', ax=axes[2])
axes[2].set_title('General_Positions attribution for ep=0.7')

plt.show()

## eps=0.8

In [ ]:
from sklearn.cluster import DBSCAN

dbScan = DBSCAN(eps=0.8, min_samples=10)
dbScan.fit(pca_position)

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(dbScan.labels_)) - (1 if -1 in dbScan.labels_ else 0)
n_noise_ = list(dbScan.labels_).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)

df['DBSCAN_08_pca'] = dbScan.labels_

In [ ]:
fig, axes = plt.subplots(3, figsize=(18, 15))

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='DBSCAN_08_pca',palette='tab10',ax=axes[0])
axes[0].set_title('Heading Accuracy vs. Def Awareness')


sns.countplot(df,x="DBSCAN_08_pca",hue="Position",palette='tab10', ax=axes[1])
axes[1].set_title('Positions attribution for ep=0.8')

sns.countplot(df,x="DBSCAN_08_pca",hue="General_Position",palette='tab10', ax=axes[2])
axes[2].set_title('General_Positions attribution for ep=0.8')

plt.show()

## eps= 0.9

In [ ]:
from sklearn.cluster import DBSCAN

dbScan = DBSCAN(eps=0.9, min_samples=10)
dbScan.fit(pca_position)

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(dbScan.labels_)) - (1 if -1 in dbScan.labels_ else 0)
n_noise_ = list(dbScan.labels_).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)

df['DBSCAN_09_pca'] = dbScan.labels_

In [ ]:
fig, axes = plt.subplots(3, figsize=(18, 15))

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='DBSCAN_09_pca',palette='tab10',ax=axes[0])
axes[0].set_title('Heading Accuracy vs. Def Awareness')


sns.countplot(df,x="DBSCAN_09_pca",hue="Position",palette='tab10', ax=axes[1])
axes[1].set_title('Positions attribution for ep=0.9')

sns.countplot(df,x="DBSCAN_09_pca",hue="General_Position",palette='tab10', ax=axes[2])
axes[2].set_title('General_Positions attribution for ep=0.9')

plt.show()

## eps= 0.3

In [ ]:
from sklearn.cluster import DBSCAN

dbScan = DBSCAN(eps=0.3, min_samples=10)
dbScan.fit(pca_position)

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(dbScan.labels_)) - (1 if -1 in dbScan.labels_ else 0)
n_noise_ = list(dbScan.labels_).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)

df['DBSCAN_03_pca'] = dbScan.labels_

In [ ]:
fig, axes = plt.subplots(3, figsize=(18, 15))

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='DBSCAN_03_pca',palette='tab10',ax=axes[0])
axes[0].set_title('Heading Accuracy vs. Def Awareness')


sns.countplot(df,x="DBSCAN_03_pca",hue="Position",palette='tab10', ax=axes[1])
axes[1].set_title('Positions attribution for ep=0.3')

sns.countplot(df,x="DBSCAN_03_pca",hue="General_Position",palette='tab10', ax=axes[2])
axes[2].set_title('General_Positions attribution for ep=0.3')

plt.show()

In [ ]:
from sklearn.cluster import DBSCAN

dbScan_3 = DBSCAN(eps=0.3, min_samples=5)
dbScan_3.fit(pca_position)

# Number of clusters in labels, ignoring noise if present.
n_clusters_ = len(set(dbScan_3.labels_)) - (1 if -1 in dbScan_3.labels_ else 0)
n_noise_ = list(dbScan_3.labels_).count(-1)

print("Estimated number of clusters: %d" % n_clusters_)
print("Estimated number of noise points: %d" % n_noise_)

In [ ]:
df['DBSCAN_03_pca_alt'] = dbScan.labels_

fig, axes = plt.subplots(3, figsize=(18, 15))

sns.scatterplot(data=df, x='Heading Accuracy', y='Def Awareness', hue='DBSCAN_03_pca_alt',palette='tab10',ax=axes[0])
axes[0].set_title('Heading Accuracy vs. Def Awareness')


sns.countplot(df,x="DBSCAN_03_pca_alt",hue="Position",palette='tab10', ax=axes[1])
axes[1].set_title('Positions attribution for ep=0.3')

sns.countplot(df,x="DBSCAN_03_pca_alt",hue="General_Position",palette='tab10', ax=axes[2])
axes[2].set_title('General_Positions attribution for ep=0.3')

plt.show()

# Conclusion:

Durante la clusterizacion con todas las variables, llegamos a obtener resultados muy disparejos y no se podia separar las posiciones de los jugadores de manera correcta. Luego redujimos la cantidad de variables y agrupamos posiciones, realizamos embeding y escalamos los datos usando MinMax en una primera instancia y luego realizamos PCA. Aplicamos el algortimo K-means y DBSCAN obteniendo mejores resultados y logrando separar las posiciones de los jugadores.
Entre K-means y DBSCAN el mejor resultado fue de K-means que nos logro separar mejor los clusters. La cantidad de clusteres que mejor representaba la posiciones de los jugadores que pueden ser sustituidos por otro fue de 4 clusteres, no observando diferencias entre las transformaciones realizadas a los datos (PCA, sin PCA). Observando que existe mezcla en los clusters entre las posiciones CM-CB y CM-ST, lo que indica que existen jugadores del medio campo con caracteristicas de delanteros o defensores que podrian jugar en esas posiciones.